# 03: SharePoint Connector for Bedrock Managed Knowledge Base

Creates a BMKB using the SharePoint Online connector with Microsoft Entra ID App-Only auth and the `Sites.Selected` scope. This is the least-privilege path and the only auth method Microsoft still supports.

### What this notebook does

1. Configures your SharePoint tenant + Entra app registrations
2. Generates a self-signed certificate for the Client App
3. Uploads the certificate to the Client App in Entra (via Microsoft Graph)
4. Grants the Client App fullcontrol on your target site (via Graph)
5. Uploads the certificate to S3 and creates the Secrets Manager secret
6. Creates the KB, IAM role, and SharePoint data source
7. Runs ingestion and queries the KB
8. Cleans up all resources

### Prerequisites

- A Microsoft 365 tenant with SharePoint Online. Note the **tenant ID** (Entra directory ID).
- Two Entra app registrations, both created in the Entra admin center:
  - **Admin App**: `Sites.FullControl.All` (Application) with admin consent, plus a client secret. Used only for Step 4 (site permission grant); delete it afterward.
  - **Client App**: `Sites.Selected` on both Microsoft Graph and SharePoint (Application) with admin consent, plus a client secret. This is the connector's identity. The certificate is uploaded in Step 3.
- One or more SharePoint site collection URLs, e.g. `https://<tenant>.sharepoint.com/sites/<site>`. The path must begin with `/sites/`, `/teams/`, or `/personal/`, not the tenant root.
- A signed-in tenant admin (for the Graph device-code flow in Step 3).
- AWS credentials with Bedrock, IAM, S3, and Secrets Manager permissions.
- OpenSSL 3.0+ locally.

### Authentication

| Auth Type | Status | When to use |
|-----------|--------|-------------|
| `ENTRA_ID_APP_ONLY` | Recommended | Default; certificate-based, least privilege |
| `OAUTH2_APP` | Retired 2026-04-02 by Microsoft (ACS deprecation). Do not use. | n/a |

### Architecture

```
SharePoint Online ──► BMKB (Bedrock-managed vector store) ──► Retrieve
      │                        │
      ├── Files & Pages        ├── IAM Role (with s3:GetObject on cert bucket)
      ├── Sites.Selected only  ├── Secrets Manager (client secret + private key)
      └── Optional filters     ├── S3 (public certificate, .pfx/.p12)
                               └── Smart Parsing
```


In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet
%pip install msal cryptography --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1: Configuration

Fill in the values from your Entra tenant and the two app registrations. `admin_app_client_secret` and `client_app_client_secret` are only shown once at Entra creation time, so copy them from the portal immediately.

In [ ]:
import boto3
import json
import sys
import time
import logging
import pprint

sys.path.insert(0, "../..")

sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# ── Configuration (update these) ─────────────────────────────────────
knowledge_base_name = f'bmkb-sharepoint-{suffix}'
knowledge_base_description = f'BMKB - SharePoint (Entra ID + Sites.Selected) - {suffix}'

# SharePoint tenant
tenant_id = '<your-microsoft-365-tenant-id>'  # UUID from Entra portal → Overview → Directory (tenant) ID
auth_type = 'ENTRA_ID_APP_ONLY'  # OAUTH2_APP is retired (2026-04-02); do not use.

# Site collection URLs to crawl. Must include /sites/, /teams/, or /personal/.
site_urls = [
    'https://<your-tenant>.sharepoint.com/sites/<your-site>',
]

# ── Entra apps ──────────────────────────────────────────────────────────
# Admin App — Sites.FullControl.All + client secret. Used ONLY for Step 4.
admin_app_client_id     = '<admin-app-client-id>'
admin_app_client_secret = '<admin-app-client-secret>'

# Client App — Sites.Selected on Graph + SharePoint. This is what the connector uses.
client_app_client_id     = '<client-app-client-id>'
client_app_client_secret = '<client-app-client-secret>'

# The Client App's Object ID (App registrations → Overview → "Object ID").
# Not the Application (client) ID — Graph's app-registration API uses object IDs.
client_app_object_id     = '<client-app-object-id>'

# ── Certificate + S3 ────────────────────────────────────────────────────
cert_common_name  = 'bmkb-sp-cert'
cert_valid_days   = 365
cert_bucket_name  = f'bmkb-sp-cert-{account_id}-{suffix}'
cert_s3_key       = 'bmkb-sharepoint/certificate.pfx'
local_key_path    = f'/tmp/bmkb-sp-{suffix}-private.key'
local_crt_path    = f'/tmp/bmkb-sp-{suffix}-certificate.crt'
local_pfx_path    = f'/tmp/bmkb-sp-{suffix}-certificate.pfx'

# Secret — set an existing ARN to skip creation
existing_secret_arn = None
secret_name = f'bmkb-sharepoint-creds-{suffix}'

# What to crawl
crawl_files = True
crawl_pages = True

# Optional filters
modified_date_after   = None   # ISO 8601 e.g. '2024-01-01T00:00:00Z'
modified_date_before  = None
inclusion_item_paths  = []     # e.g. ['/sites/mysite/Shared Documents/ProjectA']

# Models
embedding_model = 'amazon.titan-embed-text-v2:0'
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:        {region}')
print(f'Account:       {account_id}')
print(f'KB Name:       {knowledge_base_name}')
print(f'Tenant ID:     {tenant_id}')
print(f'Auth:          {auth_type}')
print(f'Sites:         {site_urls}')
print(f'Cert bucket:   {cert_bucket_name}')
print(f'Embedding:     {embedding_model}')

## Step 2: Generate the certificate

Runs `openssl` locally to produce a `.crt` (public) and `.key` (private) pair, then bundles them into a passwordless `.pfx` for Bedrock.

In [ ]:
import subprocess, os

subprocess.run([
    'openssl', 'req', '-x509', '-newkey', 'rsa:2048', '-noenc',
    '-keyout', local_key_path,
    '-out',    local_crt_path,
    '-days',   str(cert_valid_days),
    '-subj',   f'/CN={cert_common_name}',
], check=True)

# Bundle into .pfx (empty password) for Bedrock's certificateS3Path
subprocess.run([
    'openssl', 'pkcs12', '-export',
    '-inkey', local_key_path,
    '-in',    local_crt_path,
    '-out',   local_pfx_path,
    '-passout', 'pass:',
], check=True)

for path in (local_key_path, local_crt_path, local_pfx_path):
    print(f'  {path}   ({os.path.getsize(path)} bytes)')

# SHA-1 thumbprint for cross-checking against the Entra portal
subprocess.run(['openssl', 'x509', '-in', local_crt_path, '-noout', '-fingerprint', '-sha1'], check=True)

## Step 3: Upload the certificate to the Client App

Uses the Microsoft Graph device-code flow to register the certificate on your Client App. Prints a URL and short code. Open the URL in a browser, enter the code, sign in as a tenant admin, and approve the *Microsoft Graph Command Line Tools* consent prompt (a pre-registered first-party Microsoft app).

> If you'd rather do this by hand: open [Entra portal](https://entra.microsoft.com), then App registrations, your Client App, Certificates & secrets, Certificates, Upload certificate, and pick the `.crt` file at `local_crt_path`. Then skip this cell.

In [ ]:
import msal, requests, base64, hashlib

authority = f'https://login.microsoftonline.com/{tenant_id}'
GRAPH_CLI_CLIENT_ID = '14d82eec-204b-4c2f-b7e8-296a70dab67e'  # Microsoft Graph Command Line Tools

app = msal.PublicClientApplication(GRAPH_CLI_CLIENT_ID, authority=authority)
flow = app.initiate_device_flow(scopes=['Application.ReadWrite.All'])
if 'user_code' not in flow:
    raise RuntimeError(f'device flow init failed: {flow}')

print('=' * 72)
print(f'Open: {flow["verification_uri"]}')
print(f'Code: {flow["user_code"]}')
print('Sign in as a tenant admin, then wait for this cell to complete.')
print('=' * 72)

result = app.acquire_token_by_device_flow(flow)
if 'access_token' not in result:
    raise RuntimeError(f'auth failed: {result}')
graph_admin_token = result['access_token']
print('  Authenticated')

with open(local_crt_path, 'rb') as f:
    pem = f.read().decode()
der_b64 = ''.join(l for l in pem.splitlines() if not l.startswith('-----'))
der = base64.b64decode(der_b64)
thumbprint = hashlib.sha1(der).hexdigest().upper()
print(f'  Cert thumbprint: {thumbprint}')

# Merge, don't overwrite, existing keyCredentials
patch_url = f'https://graph.microsoft.com/v1.0/applications/{client_app_object_id}'
r = requests.get(patch_url + '?$select=keyCredentials',
    headers={'Authorization': f'Bearer {graph_admin_token}'})
r.raise_for_status()
existing = r.json().get('keyCredentials', [])

new_cred = {
    'type': 'AsymmetricX509Cert',
    'usage': 'Verify',
    'key': base64.b64encode(der).decode(),
    'displayName': cert_common_name,
}
r = requests.patch(patch_url,
    headers={'Authorization': f'Bearer {graph_admin_token}', 'Content-Type': 'application/json'},
    json={'keyCredentials': existing + [new_cred]})
if r.status_code >= 300:
    raise RuntimeError(f'PATCH failed [{r.status_code}]: {r.text}')
print('  Certificate added to Client App')

## Step 4: Grant the Client App `fullcontrol` on your site

`Sites.Selected` starts with zero site access. To authorize the Client App on a specific site, we call `POST /sites/{siteId}/permissions`, which itself requires `Sites.FullControl.All`. That's why we authenticate as the **Admin App** here. Microsoft designed the split so that "who can grant access" is separate from "who can use it."

Once the grant is in place, the Admin App can be deleted (Step 9).

In [ ]:
from urllib.parse import urlparse

admin_app = msal.ConfidentialClientApplication(
    client_id=admin_app_client_id,
    client_credential=admin_app_client_secret,
    authority=authority,
)
tok = admin_app.acquire_token_for_client(scopes=['https://graph.microsoft.com/.default'])
if 'access_token' not in tok:
    raise RuntimeError(f'Admin App token failed: {tok}')
admin_token = tok['access_token']
print('  Admin App token acquired')

def _resolve_site_id(url, token):
    p = urlparse(url)
    lookup = f'https://graph.microsoft.com/v1.0/sites/{p.hostname}:{p.path.rstrip("/")}'
    r = requests.get(lookup, headers={'Authorization': f'Bearer {token}'})
    r.raise_for_status()
    return r.json()['id']

def _grant_fullcontrol(site_id, client_id, token):
    url = f'https://graph.microsoft.com/v1.0/sites/{site_id}/permissions'
    payload = {
        'roles': ['fullcontrol'],
        'grantedToIdentities': [{
            'application': {'id': client_id, 'displayName': 'BMKB SharePoint Connector'}
        }],
    }
    r = requests.post(url,
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'},
        json=payload)
    if r.status_code == 409:
        return {'note': 'permission already exists'}
    r.raise_for_status()
    return r.json()

for url in site_urls:
    sid = _resolve_site_id(url, admin_token)
    grant = _grant_fullcontrol(sid, client_app_client_id, admin_token)
    print(f'  {url}')
    print(f'    site id: {sid}')
    print(f'    grant:   {grant.get("id", grant.get("note"))}')

### Sanity check: Client App can auth with the certificate

Acquire a token as the Client App using the certificate, then read the site metadata. If this fails, the KB will fail too. Fix here first, don't debug it downstream.

In [ ]:
import hashlib, msal, requests
from urllib.parse import urlparse
from cryptography import x509
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization

authority = f'https://login.microsoftonline.com/{tenant_id}'

with open(local_crt_path, 'rb') as f:
    cert = x509.load_pem_x509_certificate(f.read(), default_backend())
der = cert.public_bytes(serialization.Encoding.DER)
thumbprint = hashlib.sha1(der).hexdigest().upper()
with open(local_key_path) as f:
    private_key_pem = f.read()

client = msal.ConfidentialClientApplication(
    client_id=client_app_client_id,
    client_credential={'thumbprint': thumbprint, 'private_key': private_key_pem},
    authority=authority,
)
tok = client.acquire_token_for_client(scopes=['https://graph.microsoft.com/.default'])
if 'access_token' not in tok:
    raise RuntimeError(f'Client App cert-based auth failed: {tok}')
print('  Client App cert-based token acquired')

def _resolve_site_id(url, token):
    p = urlparse(url)
    lookup = f'https://graph.microsoft.com/v1.0/sites/{p.hostname}:{p.path.rstrip("/")}'
    r = requests.get(lookup, headers={'Authorization': f'Bearer {token}'})
    r.raise_for_status()
    return r.json()['id']

for url in site_urls:
    sid = _resolve_site_id(url, tok['access_token'])
    r = requests.get(
        f'https://graph.microsoft.com/v1.0/sites/{sid}/drives',
        headers={'Authorization': f"Bearer {tok['access_token']}"})
    r.raise_for_status()
    drives = r.json().get('value', [])
    print(f'  {url}  ->  {len(drives)} drive(s):')
    for d in drives:
        print(f'    - {d.get("name")} ({d.get("driveType")})')

## Step 5: Upload the certificate to S3 and create the Secrets Manager secret

Bedrock reads the `.pfx` from S3 during ingestion, and reads the private key and client credentials from Secrets Manager at token-acquisition time.

In [ ]:
s3 = boto3.client('s3', region_name=region)

try:
    s3.head_bucket(Bucket=cert_bucket_name)
    print(f'  Bucket exists: {cert_bucket_name}')
except s3.exceptions.ClientError:
    print(f'  Creating bucket: {cert_bucket_name}')
    if region == 'us-east-1':
        s3.create_bucket(Bucket=cert_bucket_name)
    else:
        s3.create_bucket(Bucket=cert_bucket_name,
                         CreateBucketConfiguration={'LocationConstraint': region})

s3.upload_file(local_pfx_path, cert_bucket_name, cert_s3_key)
print(f'  Uploaded: s3://{cert_bucket_name}/{cert_s3_key}')

In [ ]:
from utils.connectors.sharepoint import create_sharepoint_secret_entra_id

if existing_secret_arn:
    secret_arn = existing_secret_arn
    print(f'Using existing secret: {secret_arn}')
else:
    secret_arn = create_sharepoint_secret_entra_id(
        secret_name=secret_name,
        client_id=client_app_client_id,
        client_secret=client_app_client_secret,
        certificate_password='',   # empty because the .pfx uses no password
        private_key=private_key_pem,
        region_name=region,
    )

print(f'Secret ARN: {secret_arn}')

## Step 6: Create the KB, IAM role, and SharePoint data source

Uses raw `bedrock-agent` calls so we can:

- Explicitly set `aclEnabled: False`. The server default is `True` and requires extra Graph permissions we haven't granted.
- Pass `certificateS3Path` in the object form `{s3BucketName, s3KeyName}` the API requires.
- Attach an IAM policy for `s3:GetObject` on the cert bucket. The utility helper in this repo doesn't add this, and ingestion fails without it.

In [ ]:
bedrock_agent = boto3.client('bedrock-agent', region_name=region)
iam = boto3.client('iam')

role_name = f'AmazonBedrockExecutionRoleForKnowledgeBase_{suffix}'
trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'bedrock.amazonaws.com'},
        'Action': 'sts:AssumeRole',
        'Condition': {'StringEquals': {'aws:SourceAccount': account_id}},
    }],
}
try:
    role_arn = iam.create_role(RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy))['Role']['Arn']
    print(f'  Created role: {role_name}')
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=role_name)['Role']['Arn']
    print(f'  Role exists:  {role_name}')

policy_docs = {
    f'AmazonBedrockFoundationModelPolicyForKnowledgeBase_{suffix}': {
        'Version': '2012-10-17',
        'Statement': [{
            'Effect': 'Allow',
            'Action': ['bedrock:InvokeModel', 'bedrock:Rerank'],
            'Resource': [
                f'arn:aws:bedrock:{region}::foundation-model/{embedding_model}',
                f'arn:aws:bedrock:*::foundation-model/*',
                f'arn:aws:bedrock:{region}:{account_id}:inference-profile/*',
            ],
        }],
    },
    f'AmazonBedrockCloudWatchPolicyForKnowledgeBase_{suffix}': {
        'Version': '2012-10-17',
        'Statement': [{
            'Effect': 'Allow',
            'Action': ['logs:CreateLogGroup', 'logs:CreateLogStream', 'logs:PutLogEvents'],
            'Resource': f'arn:aws:logs:{region}:{account_id}:log-group:/aws/vendedlogs/bedrock/knowledge-base/*',
        }],
    },
    f'AmazonBedrockSecretPolicyForKnowledgeBase_{suffix}': {
        'Version': '2012-10-17',
        'Statement': [{
            'Effect': 'Allow',
            'Action': ['secretsmanager:GetSecretValue'],
            'Resource': secret_arn,
        }],
    },
    # Required for ENTRA_ID_APP_ONLY — Bedrock reads the cert from S3.
    f'AmazonBedrockCertS3PolicyForKnowledgeBase_{suffix}': {
        'Version': '2012-10-17',
        'Statement': [{
            'Effect': 'Allow',
            'Action': ['s3:GetObject', 's3:ListBucket'],
            'Resource': [
                f'arn:aws:s3:::{cert_bucket_name}',
                f'arn:aws:s3:::{cert_bucket_name}/*',
            ],
            'Condition': {'StringEquals': {'aws:ResourceAccount': [account_id]}},
        }],
    },
}
for name, doc in policy_docs.items():
    try:
        arn = iam.create_policy(PolicyName=name, PolicyDocument=json.dumps(doc))['Policy']['Arn']
        print(f'  Created policy: {name}')
    except iam.exceptions.EntityAlreadyExistsException:
        arn = f'arn:aws:iam::{account_id}:policy/{name}'
        print(f'  Policy exists:  {name}')
    iam.attach_role_policy(RoleName=role_name, PolicyArn=arn)

print('  Waiting 30s for IAM propagation...')
time.sleep(30)

In [ ]:
kb_resp = bedrock_agent.create_knowledge_base(
    name=knowledge_base_name,
    description=knowledge_base_description,
    roleArn=role_arn,
    knowledgeBaseConfiguration={
        'type': 'MANAGED',
        'managedKnowledgeBaseConfiguration': {
            'embeddingModelArn': f'arn:aws:bedrock:{region}::foundation-model/{embedding_model}',
        },
    },
)
kb_id = kb_resp['knowledgeBase']['knowledgeBaseId']
print(f'  KB ID: {kb_id}')

# Wait for the KB to reach ACTIVE. CreateKnowledgeBase is async and can take a
# minute or two, so we poll for up to ~5 min and only proceed once it's ready.
for _ in range(60):
    time.sleep(5)
    kb_state = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)['knowledgeBase']
    if kb_state['status'] in ('ACTIVE', 'FAILED'):
        break
print(f'  KB status: {kb_state["status"]}')
if kb_state['status'] != 'ACTIVE':
    raise RuntimeError(f'KB did not become ACTIVE (status={kb_state["status"]}, reasons={kb_state.get("failureReasons")})')

data_source_config = {
    'type': 'MANAGED_KNOWLEDGE_BASE_CONNECTOR',
    'managedKnowledgeBaseConnectorConfiguration': {
        'connectorParameters': {
            'type': 'SHAREPOINT',
            'version': '1',
            'aclEnabled': False,
            'connectionConfiguration': {
                'secretArn': secret_arn,
                'tenantId': tenant_id,
                'authType': auth_type,
                'certificateS3Path': {
                    's3BucketName': cert_bucket_name,
                    's3KeyName': cert_s3_key,
                },
            },
            'dataEntityConfiguration': {
                'crawlFiles': crawl_files,
                'crawlPages': crawl_pages,
                'siteUrls': site_urls,
            },
        },
        'deletionProtectionConfiguration': {'deletionProtectionStatus': 'DISABLED'},
    },
}

# Layer optional filters onto the connector params
filter_config = {}
if modified_date_after:
    filter_config['modifiedDateAfter'] = modified_date_after
if modified_date_before:
    filter_config['modifiedDateBefore'] = modified_date_before
if inclusion_item_paths:
    filter_config['inclusionItemPaths'] = inclusion_item_paths
if filter_config:
    data_source_config['managedKnowledgeBaseConnectorConfiguration']['connectorParameters']['filterConfiguration'] = filter_config

ds_resp = bedrock_agent.create_data_source(
    knowledgeBaseId=kb_id,
    name=f'{knowledge_base_name}-sp-ds',
    dataSourceConfiguration=data_source_config,
    vectorIngestionConfiguration={'parsingConfiguration': {'parsingStrategy': 'SMART_PARSING'}},
    dataDeletionPolicy='DELETE',
)
ds_id = ds_resp['dataSource']['dataSourceId']
print(f'  DS ID: {ds_id}')

for _ in range(30):
    time.sleep(10)
    ds_state = bedrock_agent.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)['dataSource']
    if ds_state['status'] in ('AVAILABLE', 'FAILED'):
        break
print(f'  DS status: {ds_state["status"]}')
if ds_state['status'] == 'FAILED':
    raise RuntimeError(f'DS creation failed: {ds_state.get("failureReasons")}')

%store kb_id
%store ds_id

## Step 7: Ingest and verify

Bedrock reaches out to Microsoft Graph, authenticates as the Client App using the certificate, and pulls file and page content from the granted sites.

In [ ]:
time.sleep(15)
job = bedrock_agent.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
job_id = job['ingestionJob']['ingestionJobId']
print(f'Ingestion job: {job_id}')

for i in range(60):
    time.sleep(10)
    j = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id
    )['ingestionJob']
    s = j.get('statistics', {})
    print(f'  [{i*10:03d}s] {j["status"]:12s} '
          f'scanned={s.get("numberOfDocumentsScanned", 0)} '
          f'indexed={s.get("numberOfNewDocumentsIndexed", 0)} '
          f'failed={s.get("numberOfDocumentsFailed", 0)}')
    if j['status'] in ('COMPLETE', 'FAILED', 'STOPPED'):
        break

if j['status'] != 'COMPLETE':
    print(f'Failure reasons: {j.get("failureReasons")}')

### Failed-document details (optional)

Reads the KB's `APPLICATION_LOGS` group. Empty output usually means everything ingested cleanly.

In [ ]:
logs_client = boto3.client('logs', region_name=region)
kb_log_group = f'/aws/vendedlogs/bedrock/knowledge-base/APPLICATION_LOGS/{kb_id}'

try:
    end_ms = int(time.time() * 1000)
    resp = logs_client.filter_log_events(
        logGroupName=kb_log_group,
        startTime=end_ms - 6 * 3600 * 1000,
        endTime=end_ms,
        filterPattern='"FAILED"',
        limit=20,
    )
    events = resp.get('events', [])
    if events:
        print(f'=== Failed Documents ({len(events)} events) ===')
        for e in events:
            log = json.loads(e['message'])
            ev = log.get('event', {})
            doc = ev.get('document_title', ev.get('document_id', 'unknown'))
            msg = ev.get('message', ev.get('status_reasons', ''))
            print(f'  {doc}: {msg}')
    else:
        print('No failed documents found in logs.')
except logs_client.exceptions.ResourceNotFoundException:
    print(f'Log group not found (KB may still be initializing): {kb_log_group}')
except Exception as e:
    print(f'Error checking logs: {e}')

## Step 8: Query the Knowledge Base

Managed KBs use `managedSearchConfiguration` on Retrieve. Some boto3 versions don't ship that shape yet, so we sign the HTTP request directly. This works regardless of local SDK version.

In [ ]:
from botocore.awsrequest import AWSRequest
from botocore.auth import SigV4Auth
from botocore.httpsession import URLLib3Session

def _retrieve(query, num_results=5):
    creds = boto3.Session().get_credentials().get_frozen_credentials()
    signer = SigV4Auth(creds, 'bedrock', region)
    req = AWSRequest(
        method='POST',
        url=f'https://bedrock-agent-runtime.{region}.amazonaws.com/knowledgebases/{kb_id}/retrieve',
        data=json.dumps({
            'retrievalQuery': {'text': query},
            'retrievalConfiguration': {'managedSearchConfiguration': {'numberOfResults': num_results}},
        }),
        headers={'Content-Type': 'application/json'},
    )
    signer.add_auth(req)
    resp = URLLib3Session().send(req.prepare())
    if resp.status_code >= 300:
        raise RuntimeError(f'HTTP {resp.status_code}: {resp.text}')
    return json.loads(resp.text)

r = _retrieve('What are the main topics in our SharePoint documentation?', num_results=5)
print('=== Retrieve API ===')
for i, res in enumerate(r.get('retrievalResults', []), 1):
    text = res['content']['text'][:120].replace('\n', ' ')
    loc = res.get('location', {}).get('sharePointLocation', {}).get('url', '')
    print(f'{i}. score={res["score"]:.4f} | {text}...')
    if loc:
        print(f'   src: {loc}')
print(f'Total: {len(r.get("retrievalResults", []))} chunks')

## Step 9: Cleanup

Only run this when you're done. Also delete the **Admin App** in Entra now. Its purpose (granting site permission via Graph) is done, and keeping it around is a standing `Sites.FullControl.All` risk.

In [ ]:
# Uncomment to delete everything created by this notebook.
# print('Deleting KB and associated resources...')
#
# # 1. Delete DS + KB
# try:
#     bedrock_agent.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
#     print(f'  Deleted DS {ds_id}')
# except Exception as e: print(f'  DS delete: {e}')
# try:
#     bedrock_agent.delete_knowledge_base(knowledgeBaseId=kb_id)
#     print(f'  Deleted KB {kb_id}')
# except Exception as e: print(f'  KB delete: {e}')
#
# # 2. Detach + delete IAM policies, then delete role
# for name in list(policy_docs.keys()):
#     arn = f'arn:aws:iam::{account_id}:policy/{name}'
#     try:
#         iam.detach_role_policy(RoleName=role_name, PolicyArn=arn)
#         iam.delete_policy(PolicyArn=arn)
#         print(f'  Deleted policy {name}')
#     except Exception as e: print(f'  {name}: {e}')
# try:
#     iam.delete_role(RoleName=role_name)
#     print(f'  Deleted role {role_name}')
# except Exception as e: print(f'  role: {e}')
#
# # 3. Delete Secrets Manager secret
# try:
#     boto3.client('secretsmanager', region_name=region).delete_secret(
#         SecretId=secret_name, ForceDeleteWithoutRecovery=True)
#     print(f'  Deleted secret {secret_name}')
# except Exception as e: print(f'  secret: {e}')
#
# # 4. Delete cert from S3 (and optionally the bucket)
# try:
#     s3.delete_object(Bucket=cert_bucket_name, Key=cert_s3_key)
#     s3.delete_bucket(Bucket=cert_bucket_name)
#     print(f'  Deleted bucket {cert_bucket_name}')
# except Exception as e: print(f'  bucket: {e}')
#
# print('Done. Also delete the Admin App in Entra: entra.microsoft.com → App registrations → bmkb-sp-admin-app → Delete')

## Summary

| What | Details |
|---|---|
| KB Type | `MANAGED`. Bedrock handles the vector store. |
| Data Source | SharePoint Online via `MANAGED_KNOWLEDGE_BASE_CONNECTOR` |
| Auth | `ENTRA_ID_APP_ONLY` with `Sites.Selected` least privilege |
| Cert | Self-signed via `openssl`, `.pfx` in S3, private key in Secrets Manager |
| Site grant | `POST /sites/{siteId}/permissions` via Graph, using the Admin App |
| Parsing | Smart Parsing (default) |
| IAM | Bedrock model, CloudWatch, Secrets Manager, and S3 (cert bucket) |

### Gotchas this notebook handles

Four things earlier drafts got wrong that trip up first-time integrators:

1. `aclEnabled` must be set to `False` explicitly. The server defaults to `True`, which needs `User.Read.All` and `GroupMember.Read.All` on Graph. Without those, the DS fails with the useless message *"Expected null"*.
2. `certificateS3Path` has to be an object of the form `{s3BucketName, s3KeyName}`, not a URI string.
3. The KB's IAM role needs `s3:GetObject` on the cert bucket. Otherwise ingestion fails with *"data source role does not have permission to read the certificate from S3"*.
4. `OAUTH2_APP` is dead. Microsoft killed the ACS flow on 2026-04-02. Only `ENTRA_ID_APP_ONLY` still works.

### SharePoint configuration options

| Parameter | Description |
|-----------|-------------|
| `site_urls` | Site collection URLs to crawl (up to 100). Path must include `/sites/`, `/teams/`, or `/personal/`. |
| `auth_type` | `ENTRA_ID_APP_ONLY` (only currently supported value). |
| `certificateS3Path` | `{s3BucketName, s3KeyName}` for the `.pfx`. |
| `aclEnabled` | Set to `False` unless the Client App has `User.Read.All` and `GroupMember.Read.All`. Immutable after creation. |
| `crawl_files` / `crawl_pages` | What to crawl. Defaults to `True`. |
| `modified_date_after` / `modified_date_before` | ISO 8601 date filters. |
| `inclusion_item_paths` | Restrict crawl to specific paths within a site. |
